In [1]:
import pandas as pd
import numpy as np

unsw = pd.read_parquet(
    "../data/processed/unsw_clean.parquet"
)

cicids = pd.read_parquet(
    "../data/processed/cicids_clean.parquet"
)

print("UNSW:", unsw.shape)
print("CICIDS:", cicids.shape)

UNSW: (2540037, 51)
CICIDS: (2824951, 80)


In [2]:
# ============================================================
# REMOVE EXACT SOURCE DUPLICATES BEFORE SPLITTING
# ============================================================

n_before = len(unsw)

duplicate_mask = unsw.duplicated()

print(
    "Exact UNSW duplicates:",
    int(duplicate_mask.sum())
)

unsw = (
    unsw
    .drop_duplicates()
    .reset_index(drop=True)
)

n_after = len(unsw)

print(
    "Rows before deduplication:",
    n_before
)

print(
    "Rows after deduplication:",
    n_after
)

print(
    "Rows removed:",
    n_before - n_after
)

assert unsw.duplicated().sum() == 0

Exact UNSW duplicates: 480633
Rows before deduplication: 2540037
Rows after deduplication: 2059404
Rows removed: 480633


### label removing

In [3]:
UNSW_LABEL_COLS = [
    "Label",
    "binary_label",
    "attack_cat",
    "attack_cat_clean",
]

y_unsw = (
    unsw["binary_label"]
    .astype("int64")
)

X_unsw = unsw.drop(
    columns=UNSW_LABEL_COLS,
    errors="ignore"
)

In [4]:
CICIDS_LABEL_COLS = [
    "Label",
    "label_clean",
    "binary_label",
]

y_cicids = (
    cicids["binary_label"]
    .astype("int64")
)

X_cicids = cicids.drop(
    columns=CICIDS_LABEL_COLS,
    errors="ignore"
)

### Drop Identifier

In [5]:
UNSW_ID_COLS = [
    "srcip",
    "dstip",
    "Stime",
    "Ltime",
]
X_unsw = X_unsw.drop(
    columns=UNSW_ID_COLS,
    errors="ignore"
)

### Phân loại categorical/ UNSW numeric

In [6]:
unsw_cat_cols = (
    X_unsw
    .select_dtypes(
        include=[
            "object",
            "string",
            "category",
        ]
    )
    .columns
    .tolist()
)

unsw_num_cols = (
    X_unsw
    .select_dtypes(
        include=np.number
    )
    .columns
    .tolist()
)

print("Categorical:", unsw_cat_cols)
print("Numeric:", len(unsw_num_cols))

Categorical: ['proto', 'state', 'service']
Numeric: 40


In [7]:
cicids_cat_cols = (
    X_cicids
    .select_dtypes(
        include=[
            "object",
            "string",
            "category",
        ]
    )
    .columns
    .tolist()
)

print(cicids_cat_cols)

[]


### Split trước scaling

In [8]:
from sklearn.model_selection import train_test_split

X_s_train, X_s_temp, y_s_train, y_s_temp = (
    train_test_split(
        X_unsw,
        y_unsw,
        test_size=0.30,
        stratify=y_unsw,
        random_state=42,
    )
)

X_s_val, X_s_test, y_s_val, y_s_test = (
    train_test_split(
        X_s_temp,
        y_s_temp,
        test_size=0.50,
        stratify=y_s_temp,
        random_state=42,
    )
)

In [9]:
target_indices = np.arange(
    len(X_cicids)
)

idx_t_train, idx_t_test = train_test_split(
    target_indices,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)

X_t_train = (
    X_cicids
    .iloc[idx_t_train]
    .reset_index(drop=True)
)

X_t_test = (
    X_cicids
    .iloc[idx_t_test]
    .reset_index(drop=True)
)

# Labels are retrieved only for final evaluation
y_t_test = (
    y_cicids
    .iloc[idx_t_test]
    .reset_index(drop=True)
)

In [10]:
# Detect constant target features using TARGET TRAIN ONLY
cicids_constant_cols = [
    col
    for col in X_t_train.columns
    if X_t_train[col].nunique(dropna=False) <= 1
]

print(
    "CICIDS constant columns:",
    cicids_constant_cols
)

# Drop exactly the same columns from train and test
X_t_train = (
    X_t_train
    .drop(columns=cicids_constant_cols)
    .copy()
)

X_t_test = (
    X_t_test
    .drop(columns=cicids_constant_cols)
    .copy()
)

assert list(X_t_train.columns) == list(X_t_test.columns)

print(
    "Target features after constant removal:",
    X_t_train.shape[1]
)

CICIDS constant columns: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Target features after constant removal: 69


### Drop constant cols ở CICIS


In [11]:
# Detect constant source features using SOURCE TRAIN ONLY
unsw_constant_cols = [
    col
    for col in X_s_train.columns
    if X_s_train[col].nunique(dropna=False) <= 1
]

print(
    "UNSW constant columns:",
    unsw_constant_cols
)

# Drop the same columns from all source splits
X_s_train = (
    X_s_train
    .drop(columns=unsw_constant_cols)
    .copy()
)

X_s_val = (
    X_s_val
    .drop(columns=unsw_constant_cols)
    .copy()
)

X_s_test = (
    X_s_test
    .drop(columns=unsw_constant_cols)
    .copy()
)

assert (
    list(X_s_train.columns)
    == list(X_s_val.columns)
    == list(X_s_test.columns)
)

# Recompute feature types AFTER final feature selection
unsw_cat_cols = (
    X_s_train
    .select_dtypes(
        include=[
            "object",
            "string",
            "category",
        ]
    )
    .columns
    .tolist()
)

unsw_num_cols = (
    X_s_train
    .select_dtypes(
        include=np.number
    )
    .columns
    .tolist()
)

print(
    "Source categorical:",
    unsw_cat_cols
)

print(
    "Source numeric:",
    len(unsw_num_cols)
)

print(
    "Source raw features:",
    X_s_train.shape[1]
)

UNSW constant columns: []
Source categorical: ['proto', 'state', 'service']
Source numeric: 40
Source raw features: 43


### Fit processing cho HDA

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from sklearn.impute import SimpleImputer

In [13]:
source_numeric_pipe = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
])

In [14]:
source_categorical_pipe = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    ),
])

In [15]:
source_preprocessor = ColumnTransformer([
    (
        "num",
        source_numeric_pipe,
        unsw_num_cols,
    ),
    (
        "cat",
        source_categorical_pipe,
        unsw_cat_cols,
    ),
])

In [16]:
X_s_train_p = source_preprocessor.fit_transform(
    X_s_train
).astype(np.float32)

X_s_val_p = source_preprocessor.transform(
    X_s_val
).astype(np.float32)

X_s_test_p = source_preprocessor.transform(
    X_s_test
).astype(np.float32)

### target preprocessing

In [17]:
target_preprocessor = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
])

In [18]:
X_t_train_p = target_preprocessor.fit_transform(
    X_t_train
).astype(np.float32)

X_t_test_p = target_preprocessor.transform(
    X_t_test
).astype(np.float32)

In [19]:
SOURCE_DIM = X_s_train_p.shape[1]
TARGET_DIM = X_t_train_p.shape[1]

LATENT_DIM = 64

print("Source input dimension:", SOURCE_DIM)
print("Target input dimension:", TARGET_DIM)
print("Shared latent dimension:", LATENT_DIM)

assert SOURCE_DIM != 0
assert TARGET_DIM != 0
assert LATENT_DIM > 0

Source input dimension: 203
Target input dimension: 69
Shared latent dimension: 64


In [20]:
assert set(UNSW_LABEL_COLS).isdisjoint(
    X_s_train.columns
), "Source label leakage detected."

assert set(CICIDS_LABEL_COLS).isdisjoint(
    X_t_train.columns
), "Target label leakage detected."


# ------------------------------------------------------------
# 2. Feature consistency across splits
# ------------------------------------------------------------

assert (
    list(X_s_train.columns)
    == list(X_s_val.columns)
    == list(X_s_test.columns)
), "Source feature mismatch across splits."

assert (
    list(X_t_train.columns)
    == list(X_t_test.columns)
), "Target feature mismatch across splits."


# ------------------------------------------------------------
# 3. No constant target features remain
# ------------------------------------------------------------

remaining_target_constants = [
    col
    for col in X_t_train.columns
    if X_t_train[col].nunique(dropna=False) <= 1
]

assert len(remaining_target_constants) == 0, (
    f"Target constant features remain: "
    f"{remaining_target_constants}"
)


# ------------------------------------------------------------
# 4. Processed arrays must contain only finite numbers
# ------------------------------------------------------------

assert np.isfinite(
    X_s_train_p
).all(), "Non-finite values in X_s_train_p."

assert np.isfinite(
    X_s_val_p
).all(), "Non-finite values in X_s_val_p."

assert np.isfinite(
    X_s_test_p
).all(), "Non-finite values in X_s_test_p."

assert np.isfinite(
    X_t_train_p
).all(), "Non-finite values in X_t_train_p."

assert np.isfinite(
    X_t_test_p
).all(), "Non-finite values in X_t_test_p."


# ------------------------------------------------------------
# 5. Check dimensions
# ------------------------------------------------------------

assert SOURCE_DIM == X_s_train_p.shape[1]
assert TARGET_DIM == X_t_train_p.shape[1]

assert SOURCE_DIM == 203, (
    f"Unexpected SOURCE_DIM: {SOURCE_DIM}"
)

assert TARGET_DIM == 69, (
    f"Unexpected TARGET_DIM: {TARGET_DIM}"
)

assert LATENT_DIM == 64


# ------------------------------------------------------------
# 6. dtype check
# ------------------------------------------------------------

assert X_s_train_p.dtype == np.float32
assert X_s_val_p.dtype == np.float32
assert X_s_test_p.dtype == np.float32

assert X_t_train_p.dtype == np.float32
assert X_t_test_p.dtype == np.float32


print("=" * 60)
print("HDA PREPARATION CHECK PASSED")
print("=" * 60)

print(
    "Source train:",
    X_s_train_p.shape
)

print(
    "Source val:",
    X_s_val_p.shape
)

print(
    "Source test:",
    X_s_test_p.shape
)

print(
    "Target adaptation:",
    X_t_train_p.shape
)

print(
    "Target test:",
    X_t_test_p.shape
)

print(
    "SOURCE_DIM:",
    SOURCE_DIM
)

print(
    "TARGET_DIM:",
    TARGET_DIM
)

print(
    "LATENT_DIM:",
    LATENT_DIM
)

HDA PREPARATION CHECK PASSED
Source train: (1441582, 203)
Source val: (308911, 203)
Source test: (308911, 203)
Target adaptation: (2259960, 69)
Target test: (564991, 69)
SOURCE_DIM: 203
TARGET_DIM: 69
LATENT_DIM: 64


In [22]:
np.save(HDA_DIR / "X_t_train.npy", X_t_train_p)
np.save(HDA_DIR / "X_t_test.npy", X_t_test_p)

np.save(
    HDA_DIR / "y_t_test.npy",
    np.asarray(y_t_test, dtype=np.int64)
)

print("Target arrays saved.")

Target arrays saved.


In [21]:
from pathlib import Path
import numpy as np

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src").is_dir() and (p / "notebooks").is_dir()),
    Path.cwd(),
)

HDA_DIR = ROOT / "data" / "processed" / "hda"
HDA_DIR.mkdir(parents=True, exist_ok=True)

# Source
np.save(HDA_DIR / "X_s_train.npy", X_s_train_p)
np.save(HDA_DIR / "X_s_val.npy", X_s_val_p)
np.save(HDA_DIR / "X_s_test.npy", X_s_test_p)

np.save(
    HDA_DIR / "y_s_train.npy",
    np.asarray(y_s_train, dtype=np.int64)
)

np.save(
    HDA_DIR / "y_s_val.npy",
    np.asarray(y_s_val, dtype=np.int64)
)

np.save(
    HDA_DIR / "y_s_test.npy",
    np.asarray(y_s_test, dtype=np.int64)
)

print("Saved source HDA arrays to:", HDA_DIR)

Saved source HDA arrays to: /Users/thonph/Desktop/KLTN/data/processed/hda
